# Fusion Evaluation Notebook

Load fused results and test set, manually align IDs, then evaluate with type-aware rules.

In [4]:
import pandas as pd
import json
from pathlib import Path

# Add PyDI to path
import sys
sys.path.insert(0, str(Path.cwd().parent.parent))

## 1. Load Data

In [5]:
# Configuration - update these paths
OUTPUT_DIR = Path("./")
FUSION_DIR = OUTPUT_DIR / "fusion"
TEST_DIR = Path("../../../usecases/input/companies/fusion")  # Adjust to your test set location

# Load fused data (from best case or specify path)
best_case_path = FUSION_DIR / "optimization" / "best_case.json"
if best_case_path.exists():
    with open(best_case_path) as f:
        best_info = json.load(f)
    best_case_dir = Path(best_info["best_case_dir"])
    print(f"Best case: {best_info['best_case_key']} (accuracy: {best_info['best_accuracy']:.1%})")
else:
    # Fallback to fused_clean.csv
    best_case_dir = FUSION_DIR

# Load fused data
fused_path = best_case_dir / "fused.csv"
if not fused_path.exists():
    fused_path = FUSION_DIR / "fused_clean.csv"
    
fused_df = pd.read_csv(fused_path)
print(f"Loaded fused data: {len(fused_df)} rows from {fused_path}")
fused_df.head()

Best case: llm_val__opt_web (accuracy: 23.1%)
Loaded fused data: 12585 rows from fusion/fused_clean.csv


,_id,_fusion_sources,country,city,id,website,name,industry,keypeople_name,assets,founded,revenue
0,fullcontact_1336,"['fullcontact_1336', 'http://www.forbes.com/co...",Australia,SydneyMLC Centre,fullcontact_1336,http://www.forbes.com/companies/gpt-group/,The GPT Group,NaN,NaN,8.400000e+09,1971-01-01,6.000000e+08
1,http://www.forbes.com/companies/encana/,"['http://www.forbes.com/companies/encana/', 'h...",Canada,Calgary,http://www.forbes.com/companies/encana/,http://www.forbes.com/companies/encana/,Encana,Integrated Oil & Gas,NaN,1.760000e+10,2002-01-01,5.700000e+09
2,http://www.forbes.com/companies/fraport/,"['http://www.forbes.com/companies/fraport/', '...",Germany,Frankfurt,http://www.forbes.com/companies/fraport/,http://www.forbes.com/companies/fraport/,Fraport,NaN,NaN,1.310000e+10,NaN,3.400000e+09
3,fullcontact_612,"['fullcontact_612', 'http://www.forbes.com/com...",United States,New York City,fullcontact_612,http://www.forbes.com/companies/pvh/,PVH Corp.,NaN,NaN,1.160000e+10,1881-01-01,8.200000e+09
4,http://dbpedia.org/resource/Marathon_Petroleum,['http://dbpedia.org/resource/Marathon_Petrole...,United States,"Findlay, Ohio",http://dbpedia.org/resource/Marathon_Petroleum,http://www.forbes.com/companies/marathon-petro...,Marathon Petroleum,Coal & Consumable Fuels,NaN,2.720000e+10,2005-01-01,8.424000e+10


In [6]:
# Load test set
from PyDI.io.loaders import load_xml

test_xml = TEST_DIR / "test_set.xml"
if test_xml.exists():
    test_df = load_xml(test_xml, nested_handling="aggregate")
    print(f"Loaded test set: {len(test_df)} rows from {test_xml}")
else:
    # Try CSV
    test_csv = TEST_DIR / "test_set.csv"
    test_df = pd.read_csv(test_csv)
    print(f"Loaded test set: {len(test_df)} rows from {test_csv}")

test_df.head()

Loaded test set: 18 rows from ../../../usecases/input/companies/fusion/test_set.xml


,id,name_provenance,name,country_provenance,country,city_provenance,city,assets_provenance,assets,revenue_provenance,revenue,founded_provenance,founded,keypeople_name
0,http://www.forbes.com/companies/avago-technolo...,fullcontact_122,Avago Technologies,fullcontact_122+fullcontact_885,United States,fullcontact_122,San Jose,http://www.forbes.com/companies/avago-technolo...,3500000000,http://www.forbes.com/companies/avago-technolo...,2700000000,,2005-01-01T00:00:00.000+01:00,NaN
1,http://www.forbes.com/companies/cp-all/,http://dbpedia.org/resource/CP_ALL,CP ALL,http://www.forbes.com/companies/cp-all/+http:/...,Thailand,http://dbpedia.org/resource/CP_ALL,Si Lom Bangkok Bang Rak District,http://www.forbes.com/companies/cp-all/,8800000000,http://www.forbes.com/companies/cp-all/,8900000000,,1988-01-01T00:00:00.000+01:00,Charoen Pokphand
2,http://www.forbes.com/companies/vale/,http://www.forbes.com/companies/vale/,Vale,http://www.forbes.com/companies/vale/+fullcont...,Brazil,fullcontact_783,Rio de Janeiro,http://www.forbes.com/companies/vale/,123700000000,http://www.forbes.com/companies/vale/,47600000000,,1872-01-01T00:00:00.000+01:00,NaN
3,http://www.forbes.com/companies/hitachi/,fullcontact_382,Hitachi,http://dbpedia.org/resource/Hitachi+http://www...,Japan,http://dbpedia.org/resource/Hitachi,"Chiyoda, Tokyo",http://www.forbes.com/companies/hitachi/,104700000000,http://www.forbes.com/companies/hitachi/,95700000000,,1910-01-01T00:00:00.000+01:00,Namihei Odaira
4,http://www.forbes.com/companies/gd-power-devel...,http://dbpedia.org/resource/GD_Power_Developme...,GD Power Development,http://dbpedia.org/resource/GD_Power_Developme...,China,http://dbpedia.org/resource/GD_Power_Developme...,Beijing,http://www.forbes.com/companies/gd-power-devel...,40400000000,http://www.forbes.com/companies/gd-power-devel...,10800000000,,1992-01-01T00:00:00.000+01:00,NaN


## 2. Inspect IDs

In [7]:
print("Fused IDs (first 20):")
print(fused_df["id"].head(20).tolist())
print(f"\nFused ID dtype: {fused_df['id'].dtype}")
print(f"Fused ID nulls: {fused_df['id'].isna().sum()}")

Fused IDs (first 20):
['fullcontact_1336', 'http://www.forbes.com/companies/encana/', 'http://www.forbes.com/companies/fraport/', 'fullcontact_612', 'http://dbpedia.org/resource/Marathon_Petroleum', 'http://www.forbes.com/companies/jbs/', 'http://www.forbes.com/companies/wpp/', 'http://www.forbes.com/companies/bnp-paribas/', 'http://www.forbes.com/companies/sumitomo-chemical/', 'http://www.forbes.com/companies/japan-tobacco/', 'fullcontact_670', 'http://www.forbes.com/companies/statoil/', 'http://www.forbes.com/companies/swiss-life-holding/', 'http://www.forbes.com/companies/saic-motor/', 'fullcontact_1327', 'http://www.forbes.com/companies/wistron/', 'http://dbpedia.org/resource/NextEra_Energy', 'http://www.forbes.com/companies/omron/', 'http://dbpedia.org/resource/Inventec', 'http://www.forbes.com/companies/viacom/']

Fused ID dtype: object
Fused ID nulls: 0


In [8]:
print("Test set IDs (first 20):")
print(test_df["id"].head(20).tolist())
print(f"\nTest ID dtype: {test_df['id'].dtype}")
print(f"Test ID nulls: {test_df['id'].isna().sum()}")

Test set IDs (first 20):
['http://www.forbes.com/companies/avago-technologies/', 'http://www.forbes.com/companies/cp-all/', 'http://www.forbes.com/companies/vale/', 'http://www.forbes.com/companies/hitachi/', 'http://www.forbes.com/companies/gd-power-development/', 'http://www.forbes.com/companies/inditex/', 'http://www.forbes.com/companies/ppg-industries/', 'http://www.forbes.com/companies/china-shenhua-energy/', 'http://www.forbes.com/companies/larsen-toubro/', 'http://www.forbes.com/companies/sherwin-williams/', 'http://www.forbes.com/companies/sistema/', 'http://www.forbes.com/companies/hewlett-packard/', 'http://www.forbes.com/companies/china-communications-services/', 'http://www.forbes.com/companies/swatch-group/', 'http://www.forbes.com/companies/abu-dhabi-commercial/', 'http://www.forbes.com/companies/l-3-communications/', 'http://www.forbes.com/companies/wells-fargo/', 'http://www.forbes.com/companies/israel-discount-bank/']

Test ID dtype: object
Test ID nulls: 0


In [9]:
# Check overlap
fused_ids = set(fused_df["id"].dropna().astype(str))
test_ids = set(test_df["id"].dropna().astype(str))

overlap = fused_ids & test_ids
print(f"Fused IDs: {len(fused_ids)}")
print(f"Test IDs: {len(test_ids)}")
print(f"Overlap: {len(overlap)}")
print(f"\nSample fused IDs not in test: {list(fused_ids - test_ids)[:5]}")
print(f"Sample test IDs not in fused: {list(test_ids - fused_ids)[:5]}")

Fused IDs: 12585
Test IDs: 18
Overlap: 5

Sample fused IDs not in test: ['fullcontact_264', 'http://dbpedia.org/resource/Texas_Health_Presbyterian_Hospital_Denton', 'http://dbpedia.org/resource/United_World_Telecom', 'fullcontact_760', 'http://dbpedia.org/resource/Girl_Candy_Films']
Sample test IDs not in fused: ['http://www.forbes.com/companies/israel-discount-bank/', 'http://www.forbes.com/companies/avago-technologies/', 'http://www.forbes.com/companies/cp-all/', 'http://www.forbes.com/companies/hewlett-packard/', 'http://www.forbes.com/companies/sistema/']


## 3. Manual ID Cleanup

Edit the cells below to fix ID alignment issues.

In [10]:
import ast

def extract_forbes_id(fusion_sources):
    """Extract Forbes ID from _fusion_sources list."""
    if pd.isna(fusion_sources) or fusion_sources is None:
        return None
    
    # Parse if it's a string representation of a list
    if isinstance(fusion_sources, str):
        try:
            sources = ast.literal_eval(fusion_sources)
        except (ValueError, SyntaxError):
            sources = [fusion_sources]
    else:
        sources = fusion_sources
    
    # Find the Forbes ID
    for src in sources:
        if isinstance(src, str) and src.startswith("http://www.forbes.com/"):
            return src
    
    return None

# Extract Forbes ID from _fusion_sources and use as the main ID
fused_df["id"] = fused_df["_fusion_sources"].apply(extract_forbes_id)

# Show how many records now have Forbes IDs
print(f"Records with Forbes ID: {fused_df['id'].notna().sum()} / {len(fused_df)}")
print(f"\nSample IDs after extraction:")
print(fused_df["id"].dropna().head(10).tolist())

Records with Forbes ID: 1931 / 12585

Sample IDs after extraction:
['http://www.forbes.com/companies/gpt-group/', 'http://www.forbes.com/companies/encana/', 'http://www.forbes.com/companies/fraport/', 'http://www.forbes.com/companies/pvh/', 'http://www.forbes.com/companies/marathon-petroleum/', 'http://www.forbes.com/companies/jbs/', 'http://www.forbes.com/companies/wpp/', 'http://www.forbes.com/companies/bnp-paribas/', 'http://www.forbes.com/companies/sumitomo-chemical/', 'http://www.forbes.com/companies/japan-tobacco/']


In [11]:
# Verify cleanup - re-check overlap
fused_ids = set(fused_df["id"].dropna().astype(str))
test_ids = set(test_df["id"].dropna().astype(str))
overlap = fused_ids & test_ids

print(f"After cleanup:")
print(f"  Overlap: {len(overlap)} / {len(test_ids)} test records")

After cleanup:
  Overlap: 17 / 18 test records


## 4. Evaluation Functions

Type-aware matching rules:
- **Strings**: Tokenized match (word overlap)
- **Dates**: Year-only comparison
- **Numbers**: 20% relative tolerance
- **Lists**: Set equality

In [ ]:
from PyDI.fusion.evaluation import tokenized_match, year_only_match, set_equality_match
import re
import numpy as np

def _is_null(val) -> bool:
    """Check if value is null/missing, handling arrays safely."""
    if val is None:
        return True
    try:
        result = pd.isna(val)
        if isinstance(result, (bool, np.bool_)):
            return bool(result)
        return False  # If array-like, not a simple null
    except (ValueError, TypeError):
        return False


def numeric_tolerance_match_relative(fused_value, expected_value, tolerance: float = 0.1) -> bool:
    """Numeric tolerance match with 10% relative tolerance (relative to expected)."""
    if fused_value is None or expected_value is None:
        return fused_value is None and expected_value is None
    
    if _is_null(fused_value) or _is_null(expected_value):
        return _is_null(fused_value) and _is_null(expected_value)
    
    try:
        fused_num = float(str(fused_value).replace(",", "").replace("$", "").strip())
        expected_num = float(str(expected_value).replace(",", "").replace("$", "").strip())
    except (ValueError, TypeError):
        return str(fused_value).strip() == str(expected_value).strip()
    
    if expected_num == 0:
        return abs(fused_num) < 1e-9
    
    relative_diff = abs(fused_num - expected_num) / abs(expected_num)
    return relative_diff <= tolerance


def infer_type(value) -> str:
    """Infer type from a value: list, date, numeric, or string."""
    if _is_null(value):
        return "string"
    
    # If it's already a list/array, return list type
    if isinstance(value, (list, tuple, np.ndarray)):
        return "list"
    
    s = str(value).strip()
    
    # Check for list
    if s.startswith("[") or ";" in s or "|" in s:
        return "list"
    
    # Check for date (yyyy-mm-dd or contains year pattern)
    if re.match(r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}", s) or re.match(r"^\d{4}$", s):
        return "date"
    
    # Check for numeric
    try:
        float(s.replace(",", "").replace("$", "").replace("%", ""))
        return "numeric"
    except ValueError:
        pass
    
    return "string"


def get_match_function(value_type: str):
    """Get the appropriate match function for a value type."""
    if value_type == "list":
        return set_equality_match
    elif value_type == "date":
        return year_only_match
    elif value_type == "numeric":
        return numeric_tolerance_match_relative
    else:
        return tokenized_match


print("Evaluation functions loaded.")

Evaluation functions loaded.


## 5. Run Evaluation

In [13]:
import numpy as np

def _is_null(val) -> bool:
    """Check if value is null/missing, handling arrays safely."""
    if val is None:
        return True
    try:
        result = pd.isna(val)
        if isinstance(result, (bool, np.bool_)):
            return bool(result)
        return False  # If array-like, not a simple null
    except (ValueError, TypeError):
        return False


def evaluate_fusion(fused_df, test_df, id_column="id", skip_columns=None):
    """Evaluate fused data against test set using type-aware matching."""
    skip_columns = skip_columns or [id_column, "_fusion_source_datasets", "_fusion_confidence", "_fusion_metadata"]
    
    # Get common attributes
    fused_attrs = set(fused_df.columns) - set(skip_columns)
    test_attrs = set(test_df.columns) - set(skip_columns)
    common_attrs = fused_attrs & test_attrs
    
    print(f"Common attributes: {sorted(common_attrs)}")
    print(f"Fused-only: {sorted(fused_attrs - test_attrs)}")
    print(f"Test-only: {sorted(test_attrs - fused_attrs)}")
    print()
    
    # Build lookup
    fused_lookup = {str(row[id_column]): row for _, row in fused_df.iterrows() if pd.notna(row[id_column])}
    
    results = {
        "total": 0,
        "correct": 0,
        "per_attribute": {attr: {"total": 0, "correct": 0} for attr in common_attrs},
        "mismatches": [],
    }
    
    for _, test_row in test_df.iterrows():
        test_id = str(test_row[id_column])
        fused_row = fused_lookup.get(test_id)
        
        if fused_row is None:
            continue  # Skip if no matching fused record
        
        for attr in common_attrs:
            expected = test_row.get(attr)
            fused = fused_row.get(attr)
            
            # Skip if both null
            if _is_null(expected) and _is_null(fused):
                continue
            
            # Infer type and get match function
            value_type = infer_type(expected)
            match_fn = get_match_function(value_type)
            
            results["total"] += 1
            results["per_attribute"][attr]["total"] += 1
            
            is_match = match_fn(fused, expected)
            
            if is_match:
                results["correct"] += 1
                results["per_attribute"][attr]["correct"] += 1
            else:
                results["mismatches"].append({
                    "id": test_id,
                    "attribute": attr,
                    "type": value_type,
                    "expected": expected,
                    "fused": fused,
                })
    
    return results


# Run evaluation
results = evaluate_fusion(fused_df, test_df)

print(f"\n{'='*60}")
print(f"OVERALL: {results['correct']}/{results['total']} = {results['correct']/results['total']:.1%}" if results['total'] > 0 else "No comparisons made")
print(f"{'='*60}")

Common attributes: ['assets', 'city', 'country', 'founded', 'keypeople_name', 'name', 'revenue']
Fused-only: ['_fusion_sources', '_id', 'industry', 'website']
Test-only: ['assets_provenance', 'city_provenance', 'country_provenance', 'founded_provenance', 'name_provenance', 'revenue_provenance']


OVERALL: 90/112 = 80.4%


In [14]:
# Per-attribute breakdown
print("\nPer-attribute accuracy:")
print("-" * 50)
for attr, stats in sorted(results["per_attribute"].items()):
    if stats["total"] > 0:
        acc = stats["correct"] / stats["total"]
        print(f"{attr:30s}: {stats['correct']:3d}/{stats['total']:3d} = {acc:.1%}")


Per-attribute accuracy:
--------------------------------------------------
assets                        :  17/ 17 = 100.0%
city                          :   7/ 17 = 41.2%
country                       :  17/ 17 = 100.0%
founded                       :  16/ 17 = 94.1%
keypeople_name                :   3/ 10 = 30.0%
name                          :  14/ 17 = 82.4%
revenue                       :  16/ 17 = 94.1%


In [15]:
# Show mismatches
print(f"\nMismatches ({len(results['mismatches'])} total):")
print("-" * 80)

mismatch_df = pd.DataFrame(results["mismatches"])
if not mismatch_df.empty:
    display(mismatch_df.head(50))
else:
    print("No mismatches!")


Mismatches (22 total):
--------------------------------------------------------------------------------


,id,attribute,type,expected,fused
0,http://www.forbes.com/companies/avago-technolo...,city,string,San Jose,New Taipei City
1,http://www.forbes.com/companies/cp-all/,city,string,Si Lom Bangkok Bang Rak District,Si LomBangkokBang Rak District
2,http://www.forbes.com/companies/vale/,founded,date,1872-01-01T00:00:00.000+01:00,1942-01-01
3,http://www.forbes.com/companies/vale/,name,string,Vale,Vale (mining company)
4,http://www.forbes.com/companies/hitachi/,city,string,"Chiyoda, Tokyo","50 Prospect Ave Tarrytown, NY"
5,http://www.forbes.com/companies/inditex/,city,string,Arteixo,A Coruña
6,http://www.forbes.com/companies/inditex/,keypeople_name,list,"[Rosalia Mera, Amancio Ortega Gaona]","["" 'Rosalía Mera']"", ""['Amancio Ortega Gaona'""]"
7,http://www.forbes.com/companies/inditex/,name,string,Inditex,INDITEX GROUP
8,http://www.forbes.com/companies/ppg-industries/,city,string,Pittsburgh,PittsburghPennsylvania
9,http://www.forbes.com/companies/ppg-industries/,keypeople_name,list,"[John Baptiste Ford, John Pitcairn]","['John Baptiste Ford', 'John Pitcairn']"


In [16]:
# Group mismatches by attribute
if not mismatch_df.empty:
    print("\nMismatches by attribute:")
    print(mismatch_df.groupby("attribute").size().sort_values(ascending=False))


Mismatches by attribute:
attribute
city              10
keypeople_name     7
name               3
founded            1
revenue            1
dtype: int64


## 6. Export Results

In [17]:
# Save mismatches for review
if not mismatch_df.empty:
    mismatch_path = FUSION_DIR / "manual_eval_mismatches.csv"
    mismatch_df.to_csv(mismatch_path, index=False)
    print(f"Saved mismatches to {mismatch_path}")

# Save summary
summary = {
    "overall_accuracy": results["correct"] / results["total"] if results["total"] > 0 else 0,
    "total_comparisons": results["total"],
    "correct_comparisons": results["correct"],
    "per_attribute": {
        attr: {
            "accuracy": stats["correct"] / stats["total"] if stats["total"] > 0 else 0,
            "total": stats["total"],
            "correct": stats["correct"],
        }
        for attr, stats in results["per_attribute"].items()
        if stats["total"] > 0
    }
}

summary_path = FUSION_DIR / "manual_eval_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Saved summary to {summary_path}")

Saved mismatches to fusion/manual_eval_mismatches.csv
Saved summary to fusion/manual_eval_summary.json
